In [6]:
%pip install requests python-dotenv pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 1.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 1.0 MB/s eta 0:00:0000:0100:010mm
Note: you may need to restart the kernel to use updated packages.


In [7]:
import requests
from typing import Any, Dict, List, Optional, Union
from dotenv import load_dotenv
import os
import json
import pandas as pd


In [2]:
def realiza_request(
    url: str,
    headers: dict = {"accept": "application/json"},
    params: dict = {}
) -> Optional[Union[Dict[str, Any], List[Any]]]:
    """Realiza uma requisição HTTP GET solicitando payload em formato JSON.

    Args:
        url (str): Endereço do recurso HTTP/HTTPS a ser consultado.

    Returns:
        Optional[Union[Dict[str, Any], List[Any]]]: Conteúdo JSON desserializado
            em caso de sucesso (status 2xx), ou None se houver falha.
    """
    error_map = {
                400: "Bad Request: Parâmetros ou estrutura de requisição inválidos.",
                401: "Unauthorized: Autenticação necessária.",
                403: "Forbidden: Acesso negado ao recurso.",
                404: "Not Found: O recurso solicitado não existe.",
                500: "Internal Server Error: Erro interno no servidor.",
                502: "Bad Gateway: Resposta inválida do servidor upstream.",
                503: "Service Unavailable: Serviço temporariamente indisponível.",
            }

    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        return response.json()

    except requests.exceptions.HTTPError as http_err:
        status_code = response.status_code
        message = error_map.get(
            status_code, f"Erro HTTP não mapeado ({status_code}): {http_err}"
        )
        print("Falha de requisição para '%s': %s", url, message)

    except requests.exceptions.Timeout:
        print("Tempo limite de requisição excedido para '%s'.", url)

    except requests.exceptions.ConnectionError:
        print("Falha na conexão de rede ao tentar acessar '%s'.", url)

    except requests.exceptions.RequestException as err:
        print("Erro inesperado na comunicação com '%s': %s", url, err)

    return None

In [26]:
load_dotenv()

URL = "https://ieeexploreapi.ieee.org/api/v1/search/articles"
APIKEY = os.getenv('IEEE_API_KEY')

search_string = r'(UAV OR UAVs OR drone OR drones OR "Internet of Drones" OR IoD OR FANET OR FANETs) AND (simulat* OR emulat* OR testbed* ) AND (“experimental setting*” OR “key internal elements” OR module OR “structural elements” OR “structural components” OR specification OR specifications OR "structural component*" OR ((UAV OR UAVs OR drone OR drones) NEAR/2 (characteristic* OR feature* OR attribut* OR propert* OR model*))) AND ("digital twin" OR "digital twins") AND  (survey OR review OR "mapping study" OR "state of the art")'

params = {
    "apikey": APIKEY,
    "querytext": search_string,
    "start_year": 2021,
    "end_year": 2026,
    "max_records": 200,
    "sort_field":"relevance",
    "sort_order":"desc"
}

res = realiza_request(URL, params=params)
res

{'total_records': 11,
 'total_searched': 7447034,
 'articles': [{'doi': '10.1109/AISEI68628.2026.11572757',
   'title': 'From Simulation to Synthesis: A Review of Modified Drone Simulators as Digital Twin and Prototyping Platforms',
   'publisher': 'IEEE',
   'isbn': '979-8-3315-7977-7',
   'rank': 1,
   'authors': {'authors': [{'affiliation': 'Research Laboratory in Applied Sciences National School of Applied Sciences, Université Mohammed Premier, Oujda, Morocco',
      'authorUrl': 'https://ieeexplore.ieee.org/author/518201911354576',
      'id': 518201911354576,
      'full_name': 'Leila Kelmoua',
      'author_order': 1,
      'authorAffiliations': {'authorAffiliation': ['Research Laboratory in Applied Sciences National School of Applied Sciences, Université Mohammed Premier, Oujda, Morocco']}},
     {'affiliation': 'Research Laboratory in Applied Sciences National School of Applied Sciences, Université Mohammed Premier, Oujda, Morocco',
      'authorUrl': 'https://ieeexplore.ieee.

In [27]:
df_res = pd.DataFrame(res['articles'])
df_res = df_res[['doi', 'title', 'issn', 'abstract', 'insert_date']]
df_res

,doi,title,issn,abstract,insert_date
0,10.1109/AISEI68628.2026.11572757,From Simulation to Synthesis: A Review of Modi...,NaN,The escalating complexity of Unmanned Aerial V...,20260626
1,10.1109/OJVT.2026.3699572,A Survey on LLM-Enabled UAV Digital Twins for ...,2644-1330,Vehicular Digital Twins (VDTs) are emerging as...,20260602
2,10.1109/TII.2024.3431106,On Constructing Efficient UAV Aerodynamic Surr...,1941-0050,Aerodynamic modeling and optimization for unma...,20240731
3,10.1109/TCCN.2026.3658762,Edge General Intelligence Through World Models...,2332-7731,Edge General Intelligence (EGI) represents a t...,20260128
4,10.1109/SoSE59841.2023.10178526,Using SysML Models as Digital Twins for Early ...,2835-3307,"For early validation of system concepts, revie...",20230717
5,10.1109/EEITE69609.2026.11679705,Integration of Digital Photogrammetry in the T...,NaN,Aerial digital photogrammetry offers a viable ...,20260916
6,10.1109/ICNS69853.2026.11570433,Creating a Digital Twin RL DAA Model to Drone ...,2155-4943,This project presents a Digital Twin framework...,20260624
7,10.1109/MIOT.2026.3661300,Adaptive Flight Planning for Neural 3D Reconst...,2576-3180,Contemporary advances in the Neural 3 Dimensio...,20260212
8,10.1109/ACCESS.2025.3648539,AI-Enabled Vehicular Data Offloading for Susta...,2169-3536,Vehicular edge computing (VEC) is emerging as ...,20251225
9,10.1109/ICUAS65942.2025.11007798,Recreation of 3D UAS Flights in High-Realism V...,2373-6720,This article presents an approach for recreati...,20250527


In [28]:
df_res.to_csv("resultados_string_12.csv", index=False)